# Cookie Cats A/B Test: Does the Paywall Gate Placement Affect Retention?

**Question:** In the mobile game Cookie Cats, a "gate" temporarily locks players unless they wait or pay. Players were randomly assigned to see this gate at **level 30** or **level 40**. Does moving the gate to level 40 actually change player retention — or is any observed difference just noise?

**Method:** This is a true randomized A/B test, so we can use statistical significance testing (chi-square for retention, t-test for engagement) to determine whether the observed differences reflect a real causal effect of gate placement, rather than random variation between the two groups.

**Structure of this notebook:**
1. Load and inspect the data
2. Compare the two groups (exploration)
3. Test for statistical significance
4. Check and handle outliers
5. Quantify the effect size
6. Conclusion and limitations

## 1. Load and Inspect the Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, ttest_ind

df = pd.read_csv("/kaggle/input/datasets/marwandiab/cookie-catsdataset/cookie_cats.csv")
df.head()

In [ ]:
df.info()

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print("\nGroup sizes:")
print(df['version'].value_counts())

In [ ]:
df.describe()

## 2. Compare the Two Groups (Summary Stats)

In [ ]:
summary = df.groupby('version').agg(
    users=('userid', 'count'),
    avg_gamerounds=('sum_gamerounds', 'mean'),
    median_gamerounds=('sum_gamerounds', 'median'),
    retention_1_rate=('retention_1', 'mean'),
    retention_7_rate=('retention_7', 'mean')
)
summary

### Bar Chart: Retention by Group

In [ ]:
retention_summary = df.groupby('version')[['retention_1', 'retention_7']].mean()

retention_summary.plot(kind='bar', figsize=(7,5), color=['#4C72B0', '#DD8452'])
plt.title('Retention Rate by Gate Version')
plt.ylabel('Retention Rate')
plt.xticks(rotation=0)
plt.legend(['1-Day Retention', '7-Day Retention'])
plt.show()

## 3. Statistical Significance Testing

### Chi-Square Test — 1-Day Retention

In [ ]:
table_1 = pd.crosstab(df['version'], df['retention_1'])
chi2_1, p_1, dof_1, expected_1 = chi2_contingency(table_1)

print("1-Day Retention Contingency Table:")
print(table_1)
print(f"\nChi-square: {chi2_1:.4f}, p-value: {p_1:.4f}")

### Chi-Square Test — 7-Day Retention

In [ ]:
table_7 = pd.crosstab(df['version'], df['retention_7'])
chi2_7, p_7, dof_7, expected_7 = chi2_contingency(table_7)

print("7-Day Retention Contingency Table:")
print(table_7)
print(f"\nChi-square: {chi2_7:.4f}, p-value: {p_7:.4f}")

## 4. Check and Handle Outliers

In [ ]:
print("Max gamerounds:", df['sum_gamerounds'].max())
df.nlargest(3, 'sum_gamerounds')

### Gamerounds Distribution — With vs. Without the Outlier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))

axes[0].hist(df['sum_gamerounds'], bins=50, color='#4C72B0')
axes[0].set_title('With Outlier (max = {:,})'.format(df['sum_gamerounds'].max()))
axes[0].set_xlabel('Game Rounds Played')
axes[0].set_ylabel('Number of Users')

df_clean_preview = df[df['sum_gamerounds'] < df['sum_gamerounds'].max()]
axes[1].hist(df_clean_preview['sum_gamerounds'], bins=50, range=(0, 500), color='#DD8452')
axes[1].set_title('Without Outlier (zoomed to 0-500 rounds)')
axes[1].set_xlabel('Game Rounds Played')

plt.tight_layout()
plt.show()

In [ ]:
# Remove the extreme outlier (one user with 49,854 rounds)
df_clean = df[df['sum_gamerounds'] < df['sum_gamerounds'].max()]

gate_30 = df_clean[df_clean['version'] == 'gate_30']['sum_gamerounds']
gate_40 = df_clean[df_clean['version'] == 'gate_40']['sum_gamerounds']

t_stat, p_val = ttest_ind(gate_30, gate_40)
print(f"T-test (outlier removed) -> t-stat: {t_stat:.4f}, p-value: {p_val:.4f}")

## 5. Effect Size — How Big Is the Real Difference?

In [ ]:
r7_30 = df[df['version']=='gate_30']['retention_7'].mean()
r7_40 = df[df['version']=='gate_40']['retention_7'].mean()

diff_pct_points = (r7_30 - r7_40) * 100
relative_drop = (r7_30 - r7_40) / r7_30 * 100

print(f"gate_30 7-day retention: {r7_30*100:.2f}%")
print(f"gate_40 7-day retention: {r7_40*100:.2f}%")
print(f"Absolute difference: {diff_pct_points:.2f} percentage points")
print(f"Relative drop: {relative_drop:.2f}%")

## 6. Conclusion

Moving the paywall gate from level 30 to level 40 caused a statistically significant drop in 7-day retention (p = 0.0016), from 19.02% to 18.20% — a 0.82 percentage point (4.3% relative) decline. 1-day retention showed a smaller difference that was **not** statistically significant (p = 0.0755). Average engagement (game rounds played) showed no meaningful difference between groups once a single extreme outlier was removed (p = 0.9495).

**Bottom line:** the gate change appears to genuinely hurt longer-term (7-day) retention, even though it doesn't meaningfully affect short-term (1-day) retention or overall engagement. This suggests players who hit the gate later (level 40) are somewhat more likely to churn before the 7-day mark than those who hit it earlier (level 30).

### Limitations
- This was a randomized A/B test, so causal inference is on solid ground — the two groups should be comparable by design, and no diff-in-diff or control group is needed.
- One extreme outlier (49,854 game rounds from a single user) was identified and removed before comparing average engagement, to avoid a skewed result.
- The effect size is small in absolute terms (0.82 percentage points); whether a 4.3% relative drop in retention matters commercially depends on business context and scale (e.g., at millions of daily active users, even small percentage-point shifts translate into large absolute numbers of lost players).
- The data only tells us **that** retention dropped, not **why** — we don't have information on player sentiment, in-game purchases, or session-level behavior that could explain the mechanism behind the drop.
- Results reflect this specific game and gate mechanic; they may not generalize to other games or monetization gate designs.